[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/learn-python-from-scratch-to-advanced/blob/main/notebooks/06-dsa-interview/03_sliding-window.ipynb)

*DSA Interview Handbook · Notebook 3 of 14*

# Sliding Window

Grow a window on the right, shrink it on the left, and never re-scan what's inside.

### 👩‍🏫 From your teacher

**The picture to keep in your head:** A window slides over the data like a moving train carriage: add what enters, remove what leaves.

**What interviewers really test here:** Fixed vs variable windows, and when negatives break the method.

### How to use this notebook

1. **Run the setup cell** just below.
2. **Read each lesson slowly.** Work the numbers on paper before reading on.
3. **Run and then change the examples.** Breaking code on purpose is how you learn.
4. **Try each practice question before opening an answer.** If you're stuck, open one hint at a time.
5. **Say your interview answers out loud.** Knowing something and explaining it are different skills.

### Your roadmap

- [ ] 1. Subarrays, and why brute force wastes work *(Basics)*
- [ ] 2. Variable-size window: expand, then shrink *(Core)*
- [ ] 3. Windows that count characters *(Core)*
- [ ] 4. Counting subarrays: the atMost trick *(Advanced)*
- [ ] 5. Window maximum with a monotonic deque *(Advanced)*
- [ ] 20 practice questions
- [ ] Interview questions, out loud

In [1]:
# ▶ Run this cell first. It prepares the notebook; you don't need to read it.
import contextlib, copy, io, math, sys
import numpy as np, matplotlib.pyplot as plt
%matplotlib inline

PLAIN = False

class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def mk_list(vals, pos=-1):
    dummy = cur = ListNode()
    nodes = []
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
        nodes.append(cur)
    if pos >= 0 and nodes:
        nodes[-1].next = nodes[pos]
    return dummy.next

def list_arr(head):
    out, seen = [], set()
    while head and id(head) not in seen:
        seen.add(id(head))
        out.append(head.val)
        head = head.next
    return out

def mk_tree(vals):
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = [root], 1
    while queue and i < len(vals):
        node = queue.pop(0)
        for side in ("left", "right"):
            if i < len(vals) and vals[i] is not None:
                child = TreeNode(vals[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root

def tree_arr(root):
    out, queue = [], [root]
    while queue:
        node = queue.pop(0)
        if node:
            out.append(node.val)
            queue += [node.left, node.right]
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out

def find(root, val):
    if not root or root.val == val:
        return root
    return find(root.left, val) or find(root.right, val)

ListNode.__repr__ = lambda self: f"ListNode{list_arr(self)}"
TreeNode.__repr__ = lambda self: f"TreeNode{tree_arr(self)}"

def plain(x, digits=4):
    """NumPy / pandas results as plain Python values, so they compare with a test's expected answer."""
    np = sys.modules.get("numpy")
    pd = sys.modules.get("pandas")
    if pd is not None:
        if isinstance(x, pd.DataFrame):
            return {str(c): plain(x[c].tolist(), digits) for c in x.columns}
        if isinstance(x, pd.Series):
            return {plain(k, digits): plain(v, digits) for k, v in x.items()}
        if isinstance(x, pd.Index):
            return plain(x.tolist(), digits)
        if x is pd.NaT:
            return None
        if isinstance(x, pd.Timestamp):
            return str(x.date()) if x == x.normalize() else str(x)
    if np is not None:
        if isinstance(x, np.ndarray):
            return plain(x.tolist(), digits)
        if isinstance(x, np.generic):
            return plain(x.item(), digits)
    if isinstance(x, float):
        return None if x != x else round(x, digits) + 0.0
    if isinstance(x, list):
        return [plain(v, digits) for v in x]
    if isinstance(x, tuple):
        return tuple(plain(v, digits) for v in x)
    if isinstance(x, dict):
        return {plain(k, digits): plain(v, digits) for k, v in x.items()}
    if isinstance(x, (set, frozenset)):
        return {plain(v, digits) for v in x}
    return x


def check(fn, tests, plain_compare=None, listify=False, design=False, prep=None, post=None,
          norm=None, verify=None, runner=None, inplace=None):
    """Run your code against the tests and show which pass."""
    use_plain = PLAIN if plain_compare is None else plain_compare
    passed, problems = 0, []
    for args, want in tests:
        try:
            a = copy.deepcopy(args)
            if design:
                obj = fn(*a[0])
                got = [getattr(obj, m)(*p) for m, p in a[1]]
            elif runner is not None:
                got = runner(fn, a)
            else:
                if prep is not None:
                    a = prep(a)
                with contextlib.redirect_stdout(io.StringIO()):
                    got = fn(*a)
                if inplace is not None:
                    got = a[inplace]
                if listify:
                    got = list(got)
                if post is not None:
                    got = post(got)
            if verify is not None:
                ok = bool(verify(copy.deepcopy(args), got))
            elif norm is not None:
                ok = norm(copy.deepcopy(got)) == norm(copy.deepcopy(want))
            else:
                ok = (plain(got), plain(want)) if use_plain else (got, want)
                ok = ok[0] == ok[1]
            ok = ok if isinstance(ok, bool) else False
        except Exception as e:  # your code raised an error: report it, don't stop the notebook
            ok, got = False, f"{type(e).__name__}: {e}"
        passed += ok
        if not ok:
            problems.append((args, want, got))
    plt.close("all")
    if passed == len(tests):
        print(f"✅ All {len(tests)} tests passed. Well done! Now compare with the answers below.")
    else:
        print(f"❌ {passed}/{len(tests)} tests passed. First one to fix:")
        args, want, got = problems[0]
        print("   input:   ", repr(args)[:300])
        print("   expected:", repr(want)[:300])
        print("   you got: ", repr(got)[:300])
    return None

print("Ready! Scroll down and start learning.")

Ready! Scroll down and start learning.


---
# Part 1: The theory in one place

Read this once for the big picture. It's fine if it doesn't all stick yet: the lessons will make it concrete, and you can come back to revise.

### Definition

The **sliding window technique** processes a contiguous range `[l, r]` of an array or string — the **window** — and moves it across the input, updating a summary of the window (sum, counts, max) **incrementally** instead of recomputing it from scratch.



Each element enters the window once (when `r` passes it) and leaves once (when `l` passes it), giving O(n) total time for problems that naively take O(n²) or O(n · k).

### Types of windows

| Type | Size | Movement rule | Examples |
| --- | --- | --- | --- |
| **Fixed-size** | always k | add a[r], remove a[r − k] | max sum / average of size k, anagram search, vowels in substring |
| **Variable-size (longest)** | changes | expand r; shrink l while invalid; record after shrinking | longest substring without repeats, max consecutive ones with k flips |
| **Variable-size (shortest)** | changes | expand r; while valid, record then shrink | minimum size subarray sum, minimum window substring |
| **Counting** | changes | after shrinking, add r − l + 1 | subarrays with at most / exactly K … |

### When the technique applies: monotonicity

Shrinking from the left is only correct if the window's validity is **monotonic**:




- If window [l, r] is **invalid** because it is "too much", then every larger window [l', r] with l' &lt; l is also invalid. So once we move l forward we never need to move it back.
- Example: with **non-negative** numbers, "sum ≤ k" is monotonic — removing elements can only reduce the sum.
- Counter-example: with **negative** numbers, removing an element can *increase* the sum, so the window cannot decide when to shrink. Use **prefix sums + hash map** instead (Subarray Sum Equals K).

### Complexity analysis

A variable window has a `while` loop inside a `for` loop, which looks like O(n²). It is O(n) because `l` only increases and never exceeds n, so the inner loop runs at most n times **in total** across the whole algorithm (aggregate / amortised analysis).



| Window state | Update per step | Space |
| --- | --- | --- |
| sum / count of a condition | O(1) | O(1) |
| character counts | O(1) | O(σ), σ = alphabet size |
| max / min | O(1) amortised with a monotonic deque | O(k) |
| median | O(log k) with two heaps | O(k) |

### Advantages and limitations

**👍 Pros**

##### Advantages

- Turns O(n²) / O(nk) into O(n)
- Single pass; works on streams
- Small memory: only the window's summary



**👎 Cons**

##### Limitations

- Only for **contiguous** ranges (not subsequences)
- Needs a monotonic validity condition
- "Exactly K" needs the atMost(K) − atMost(K−1) trick

### Applications

- **Networking**: TCP's sliding window controls how many packets are in flight.
- **Rate limiting**: count requests in the last 60 seconds.
- **Signal processing / finance**: moving averages, rolling max/min of prices.
- **Text processing**: finding the shortest snippet containing all search keywords.
- **Monitoring**: alerts when the error count in the last N events crosses a threshold.

---
# Part 2: Lessons, step by step

## Lesson 1: Subarrays, and why brute force wastes work

*Level: Basics*

A **subarray** (or substring) is a *contiguous* piece: `[3, 1, 4]` inside `[2, 3, 1, 4, 5]`. An array of n elements has about n²/2 subarrays.



Brute force recomputes each one from scratch. But neighbouring windows overlap almost completely:



```text
[2, 3, 1, 4, 5]
 └──┬──┘           window 0..2  sum = 6
    └──┬──┘        window 1..3  sum = 6 − 2 + 4 = 8
```



A **sliding window** reuses the overlap: when the window moves one step, **add** the element that enters and **remove** the one that leaves. Each step is O(1), so the whole scan is O(n).

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [2]:
def fixed_window_lesson():
    nums, k = [2, 3, 1, 4, 5, 1], 3
    total = sum(nums[:k])
    best = total
    for r in range(k, len(nums)):
        total += nums[r] - nums[r - k]
        best = max(best, total)
    return best


fixed_window_lesson()

10

<details>
<summary><b>🔎 What happens, step by step</b></summary>

1. First window sum = 6
2. Slide: +4 enters, −2 leaves → 8
3. Slide: +5 enters, −3 leaves → 10
4. Slide: +1 enters, −1 leaves → 10

</details>


**✏️ Your turn:** you now have what you need for [Q1](#q1), [Q14](#q14). Try one now, or carry on and do them in Part 3.

## Lesson 2: Variable-size window: expand, then shrink

*Level: Core*

When the window size isn't fixed, use two pointers `l` and `r`:




1. Move `r` forward one step and add `a[r]` to the window.
2. **While** the window breaks the rule, remove `a[l]` and move `l` forward.
3. Now the window is valid — update the answer.



```python
l = 0
for r in range(len(a)):
    add(a[r])
    while invalid():
        remove(a[l]); l += 1
    best = max(best, r - l + 1)
```



It's still O(n): `l` and `r` each move forward at most n times in total, even though there's a loop inside a loop.



**💡 Tip:** It only works when the rule is **monotonic**: shrinking a bad window can make it good, and growing a good window can only make it worse (or vice versa). Positive numbers with "sum ≥ target" qualify; negative numbers don't.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [3]:
def variable_window_lesson():
    nums, target = [2, 3, 1, 2, 4, 3], 7
    l = total = 0
    best = float("inf")
    for r, x in enumerate(nums):
        total += x
        while total >= target:
            best = min(best, r - l + 1)
            total -= nums[l]
            l += 1
    return best


variable_window_lesson()

2

<details>
<summary><b>🔎 What happens, step by step</b></summary>

1. Expand: add 2 → sum 2
2. Expand: add 3 → sum 5
3. Expand: add 1 → sum 6
4. Expand: add 2 → sum 8
5. Sum 8 ≥ 7: record length 4, then shrink
6. Expand: add 4 → sum 10
7. Sum 10 ≥ 7: record length 4, then shrink
8. Sum 7 ≥ 7: record length 3, then shrink
9. Expand: add 3 → sum 9
10. Sum 9 ≥ 7: record length 3, then shrink
11. Sum 7 ≥ 7: record length 2, then shrink

</details>


**✏️ Your turn:** you now have what you need for [Q6](#q6), [Q3](#q3), [Q7](#q7), [Q8](#q8), [Q16](#q16), [Q17](#q17). Try one now, or carry on and do them in Part 3.

## Lesson 3: Windows that count characters

*Level: Core*

Many string problems ask about the *letters* inside a window. Keep a count dict for the window and update it as letters enter and leave.




- **Permutation / anagram in a string**: fixed window of length |p|; compare its counts with p's counts.
- **Minimum window substring**: expand until every needed letter is covered, then shrink while still covered. Track `have` (letters whose count is satisfied) vs `need` so the check is O(1), not a full dict comparison.
- **Longest repeating replacement**: window is valid if `length − (count of its most common letter) ≤ k`.



**⚠️ Watch out:** When a count drops to 0, delete the key (`del count[c]`) if you use `len(count)` as "number of distinct letters".

**✏️ Your turn:** you now have what you need for [Q5](#q5), [Q9](#q9), [Q18](#q18), [Q4](#q4), [Q10](#q10). Try one now, or carry on and do them in Part 3.

## Lesson 4: Counting subarrays: the atMost trick

*Level: Advanced*

"How many subarrays have **exactly** K distinct values / K odd numbers / sum K?" is hard to do with one window, because a valid window can become invalid and valid again.



"**At most** K" is easy: for each `r`, after shrinking, every start from `l` to `r` gives a valid subarray ending at `r` — that's `r − l + 1` new subarrays.



```python
exactly(K) = atMost(K) − atMost(K − 1)
```



This one identity solves a whole family of problems in O(n).

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [4]:
def at_most_lesson():
    nums, k = [1, 2, 1, 3], 2

    def at_most(k):
        count, l, total = {}, 0, 0
        for r, x in enumerate(nums):
            count[x] = count.get(x, 0) + 1
            while len(count) > k:
                count[nums[l]] -= 1
                if count[nums[l]] == 0:
                    del count[nums[l]]
                l += 1
            total += r - l + 1
        return total

    return at_most(k) - at_most(k - 1)


at_most_lesson()

4

<details>
<summary><b>🔎 What happens, step by step</b></summary>

1. atMost(2): subarrays ending at 0 are those starting at 0..0 → +1
2. atMost(2): subarrays ending at 1 are those starting at 0..1 → +2
3. atMost(2): subarrays ending at 2 are those starting at 0..2 → +3
4. atMost(2): subarrays ending at 3 are those starting at 2..3 → +2
5. atMost(1): subarrays ending at 0 are those starting at 0..0 → +1
6. atMost(1): subarrays ending at 1 are those starting at 1..1 → +1
7. atMost(1): subarrays ending at 2 are those starting at 2..2 → +1
8. atMost(1): subarrays ending at 3 are those starting at 3..3 → +1

</details>


**✏️ Your turn:** you now have what you need for [Q20](#q20), [Q12](#q12), [Q13](#q13). Try one now, or carry on and do them in Part 3.

## Lesson 5: Window maximum with a monotonic deque

*Level: Advanced*

Keeping the **max** of a sliding window is harder than keeping a sum — when the max leaves, what's the next max? Recomputing is O(k) per step.



Keep a `deque` of indices whose values are **decreasing** from front to back:




- Before adding `i`, pop from the back every index whose value ≤ `a[i]`. They can never be a max again — `a[i]` is newer and at least as big.
- Pop the front if it has left the window.
- The front is always the window max.



Each index enters and leaves the deque once → O(n) total. The same idea (a monotonic structure) returns in the Stack topic.

**✏️ Your turn:** you now have what you need for [Q19](#q19), [Q2](#q2). Try one now, or carry on and do them in Part 3.

---
# Part 3: Practice like an interview

There are 20 questions here, easiest first.

**Work the way you would in a real interview:** restate the problem, try an example by hand, say the simple approach and its cost, improve it, then write the code. Test it with the cell. Only open an answer once you've made a real attempt, because struggling is where the learning happens.

<a id="q1"></a>
### Q1. Maximum Average Subarray I

🟢 Warm-up · *Fixed window · sum*

Return the maximum average of any contiguous subarray of length `k`.

**Examples**

- `nums = [1, 12, -5, -6, 50, 3], k = 4` → `12.75`
- `nums = [5], k = 1` → `5.0`

In [ ]:
def find_max_average(nums, k):
    """
    Return the maximum average of any contiguous subarray of length k.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(find_max_average, [
    (([1, 12, -5, -6, 50, 3], 4), 12.75),
    (([5], 1), 5.0),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Sum every window of length k from scratch.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Keep the window sum: add the new element, subtract the one that falls out.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Sum every window of length k from scratch.

```python
def find_max_average(nums, k):
    return max(sum(nums[i:i + k]) for i in range(len(nums) - k + 1)) / k
```

**Cost:** O(n · k) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Keep the window sum: add the new element, subtract the one that falls out.

```python
def find_max_average(nums, k):
    total = best = sum(nums[:k])
    for r in range(k, len(nums)):
        total += nums[r] - nums[r - k]
        best = max(best, total)
    return best / k
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Sum every window of length k from scratch. That costs O(n · k) time and O(1) space."

"I can do better: Keep the window sum: add the new element, subtract the one that falls out. That's O(n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `nums = [1, 12, -5, -6, 50, 3], k = 4`

1. First window sum = 2
2. +50 −1 → sum 51
3. +3 −12 → sum 42

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/sliding-window/maximum-average-subarray-i)

<a id="q2"></a>
### Q2. Contains Duplicate II

🟢 Warm-up · *Window as a set*

Return `True` if two equal values sit at most `k` indices apart.

**Examples**

- `nums = [1, 2, 3, 1], k = 3` → `True`
- `nums = [1, 0, 1, 1], k = 1` → `True`
- `nums = [1, 2, 3, 1, 2, 3], k = 2` → `False`

In [ ]:
def contains_nearby_duplicate(nums, k):
    """
    Return True if two equal values sit at most k indices apart.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(contains_nearby_duplicate, [
    (([1, 2, 3, 1], 3), True),
    (([1, 0, 1, 1], 1), True),
    (([1, 2, 3, 1, 2, 3], 2), False),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Compare each element with the next k elements.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Keep a set of the last k values. If nums[i] is in it, a close duplicate exists; otherwise add it and evict nums[i − k].

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Compare each element with the next k elements.

```python
def contains_nearby_duplicate(nums, k):
    for i in range(len(nums)):
        for j in range(i + 1, min(len(nums), i + k + 1)):
            if nums[i] == nums[j]:
                return True
    return False
```

**Cost:** O(n · k) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Keep a set of the last k values. If nums[i] is in it, a close duplicate exists; otherwise add it and evict nums[i − k].

```python
def contains_nearby_duplicate(nums, k):
    window = set()
    for i, x in enumerate(nums):
        if x in window:
            return True
        window.add(x)
        if len(window) > k:
            window.remove(nums[i - k])
    return False
```

**Cost:** O(n) time, O(k) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Compare each element with the next k elements. That costs O(n · k) time and O(1) space."

"I can do better: Keep a set of the last k values. If nums[i] is in it, a close duplicate exists; otherwise add it and evict nums[i − k]. That's O(n) time and O(k) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `nums = [1, 2, 3, 1], k = 3`

1. Is 1 among the last 3 values?
2. Is 2 among the last 3 values?
3. Is 3 among the last 3 values?
4. Is 1 among the last 3 values?

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/sliding-window/contains-duplicate-ii)

<a id="q3"></a>
### Q3. Longest Substring Without Repeating Characters

🟡 Interview standard · *Variable window · last index*

Return the length of the longest substring of `s` with no repeated characters.

**Examples**

- `s = 'abcabcbb'` → `3`
- `s = 'bbbbb'` → `1`
- `s = 'pwwkew'` → `3`

In [ ]:
def length_of_longest_substring(s):
    """
    Return the length of the longest substring of s with no repeated characters.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(length_of_longest_substring, [
    (('abcabcbb',), 3),
    (('bbbbb',), 1),
    (('pwwkew',), 3),
    (('',), 0),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Check every substring for duplicates with a set.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Grow r. Store each character's last index; if s[r] was seen inside the window, jump l to one past that index. Record r − l + 1.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Check every substring for duplicates with a set.

```python
def length_of_longest_substring(s):
    best = 0
    for i in range(len(s)):
        for j in range(i, len(s)):
            if len(set(s[i:j + 1])) == j - i + 1:
                best = max(best, j - i + 1)
    return best
```

**Cost:** O(n³) time, O(min(n, σ)) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Grow r. Store each character's last index; if s[r] was seen inside the window, jump l to one past that index. Record r − l + 1.

```python
def length_of_longest_substring(s):
    last = {}
    l = best = 0
    for r, ch in enumerate(s):
        if ch in last and last[ch] >= l:
            l = last[ch] + 1
        last[ch] = r
        best = max(best, r - l + 1)
    return best
```

**Cost:** O(n) time, O(σ) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Check every substring for duplicates with a set. That costs O(n³) time and O(min(n, σ)) space."

"I can do better: Grow r. Store each character's last index; if s[r] was seen inside the window, jump l to one past that index. Record r − l + 1. That's O(n) time and O(σ) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `s = 'pwwkew'`

1. Window 'p' has no repeats; best = 1
2. Window 'pw' has no repeats; best = 2
3. Window 'w' has no repeats; best = 2
4. Window 'wk' has no repeats; best = 2
5. Window 'wke' has no repeats; best = 3
6. Window 'kew' has no repeats; best = 3

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/sliding-window/longest-substring-without-repeating-characters)

<a id="q4"></a>
### Q4. Longest Repeating Character Replacement

🟡 Interview standard · *Variable window · max count*

You may replace at most `k` characters of `s`. Return the length of the longest substring of one repeated letter you can get.

**Examples**

- `s = 'ABAB', k = 2` → `4`
- `s = 'AABABBA', k = 1` → `4`

In [ ]:
def character_replacement(s, k):
    """
    You may replace at most k characters of s. Return the length of the longest
    substring of one repeated letter you can get.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(character_replacement, [
    (('ABAB', 2), 4),
    (('AABABBA', 1), 4),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

For every substring, cost = length − count of its most common letter; keep the longest with cost ≤ k.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Slide a window keeping letter counts and the highest count seen. If length − maxCount &gt; k, too many letters need replacing → shrink once from the left.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** For every substring, cost = length − count of its most common letter; keep the longest with cost ≤ k.

```python
def character_replacement(s, k):
    best = 0
    for i in range(len(s)):
        count = {}
        for j in range(i, len(s)):
            count[s[j]] = count.get(s[j], 0) + 1
            if j - i + 1 - max(count.values()) <= k:
                best = max(best, j - i + 1)
    return best
```

**Cost:** O(n²) time, O(σ) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Slide a window keeping letter counts and the highest count seen. If length − maxCount &gt; k, too many letters need replacing → shrink once from the left.

```python
def character_replacement(s, k):
    count, l, maxf, best = {}, 0, 0, 0
    for r, ch in enumerate(s):
        count[ch] = count.get(ch, 0) + 1
        maxf = max(maxf, count[ch])
        if r - l + 1 - maxf > k:
            count[s[l]] -= 1
            l += 1
        best = max(best, r - l + 1)
    return best
```

**Cost:** O(n) time, O(σ) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: For every substring, cost = length − count of its most common letter; keep the longest with cost ≤ k. That costs O(n²) time and O(σ) space."

"I can do better: Slide a window keeping letter counts and the highest count seen. If length − maxCount &gt; k, too many letters need replacing → shrink once from the left. That's O(n) time and O(σ) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `s = 'ABAB', k = 2`

1. Window length 1, most common letter appears 1× → replace 0 ≤ k
2. Window length 2, most common letter appears 1× → replace 1 ≤ k
3. Window length 3, most common letter appears 2× → replace 1 ≤ k
4. Window length 4, most common letter appears 2× → replace 2 ≤ k

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/sliding-window/longest-repeating-character-replacement)

<a id="q5"></a>
### Q5. Permutation in String

🟡 Interview standard · *Fixed window · counts*

Return `True` if some permutation of `s1` appears as a substring of `s2`.

**Examples**

- `s1 = 'ab', s2 = 'eidbaooo'` → `True`
- `s1 = 'ab', s2 = 'eidboaoo'` → `False`

In [ ]:
def check_inclusion(s1, s2):
    """
    Return True if some permutation of s1 appears as a substring of s2.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(check_inclusion, [
    (('ab', 'eidbaooo'), True),
    (('ab', 'eidboaoo'), False),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

For every window of length |s1| in s2, sort it and compare to sorted(s1).

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Keep a fixed window of length |s1| sliding over s2 with letter counts. Compare the 26-count arrays (or track how many letters match) after each slide.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** For every window of length |s1| in s2, sort it and compare to sorted(s1).

```python
def check_inclusion(s1, s2):
    m, target = len(s1), sorted(s1)
    return any(sorted(s2[i:i + m]) == target for i in range(len(s2) - m + 1))
```

**Cost:** O(n · m log m) time, O(m) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Keep a fixed window of length |s1| sliding over s2 with letter counts. Compare the 26-count arrays (or track how many letters match) after each slide.

```python
def check_inclusion(s1, s2):
    m = len(s1)
    if m > len(s2):
        return False
    need, win = [0] * 26, [0] * 26
    for ch in s1:
        need[ord(ch) - 97] += 1
    for r, ch in enumerate(s2):
        win[ord(ch) - 97] += 1
        if r >= m:
            win[ord(s2[r - m]) - 97] -= 1
        l = max(0, r - m + 1)
        if win == need:
            return True
    return False
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: For every window of length |s1| in s2, sort it and compare to sorted(s1). That costs O(n · m log m) time and O(m) space."

"I can do better: Keep a fixed window of length |s1| sliding over s2 with letter counts. Compare the 26-count arrays (or track how many letters match) after each slide. That's O(n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `s1 = 'abc', s2 = 'ebcbaco'`

1. Window 'e' — same letter counts as 'abc'? False
2. Window 'eb' — same letter counts as 'abc'? False
3. Window 'ebc' — same letter counts as 'abc'? False
4. Window 'bcb' — same letter counts as 'abc'? False
5. Window 'cba' — same letter counts as 'abc'? True

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/sliding-window/permutation-in-string)

<a id="q6"></a>
### Q6. Minimum Size Subarray Sum

🟡 Interview standard · *Variable window · sum*

All values are positive. Return the length of the shortest subarray with sum ≥ `target`, or 0.

**Examples**

- `target = 7, nums = [2, 3, 1, 2, 4, 3]` → `2`
- `target = 4, nums = [1, 4, 4]` → `1`
- `target = 11, nums = [1, 1, 1, 1, 1, 1, 1, 1]` → `0`

In [ ]:
def min_sub_array_len(target, nums):
    """
    All values are positive. Return the length of the shortest subarray with sum
    ≥ target, or 0.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(min_sub_array_len, [
    ((7, [2, 3, 1, 2, 4, 3]), 2),
    ((4, [1, 4, 4]), 1),
    ((11, [1, 1, 1, 1, 1, 1, 1, 1]), 0),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Fix a start and extend until the sum reaches target.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Add nums[r] to the window sum. While the sum is at least target, record the length and drop nums[l]. Positive numbers make shrinking safe.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Fix a start and extend until the sum reaches target.

```python
def min_sub_array_len(target, nums):
    best = 0
    for i in range(len(nums)):
        total = 0
        for j in range(i, len(nums)):
            total += nums[j]
            if total >= target:
                if not best or j - i + 1 < best:
                    best = j - i + 1
                break
    return best
```

**Cost:** O(n²) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Add nums[r] to the window sum. While the sum is at least target, record the length and drop nums[l]. Positive numbers make shrinking safe.

```python
def min_sub_array_len(target, nums):
    l = total = 0
    best = float("inf")
    for r, x in enumerate(nums):
        total += x
        while total >= target:
            best = min(best, r - l + 1)
            total -= nums[l]
            l += 1
    return 0 if best == float("inf") else best
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Fix a start and extend until the sum reaches target. That costs O(n²) time and O(1) space."

"I can do better: Add nums[r] to the window sum. While the sum is at least target, record the length and drop nums[l]. Positive numbers make shrinking safe. That's O(n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `target = 7, nums = [2, 3, 1, 2, 4, 3]`

1. Sum 8 ≥ 7: length 4. Shrink.
2. Sum 10 ≥ 7: length 4. Shrink.
3. Sum 7 ≥ 7: length 3. Shrink.
4. Sum 9 ≥ 7: length 3. Shrink.
5. Sum 7 ≥ 7: length 2. Shrink.

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/sliding-window/minimum-size-subarray-sum)

<a id="q7"></a>
### Q7. Max Consecutive Ones III

🟡 Interview standard · *Variable window · budget*

`nums` is binary. You may flip at most `k` zeros. Return the longest run of 1s possible.

**Examples**

- `nums = [1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0], k = 2` → `6`
- `nums = [0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1], k = 3` → `10`

In [ ]:
def longest_ones(nums, k):
    """
    nums is binary. You may flip at most k zeros. Return the longest run of 1s
    possible.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(longest_ones, [
    (([1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0], 2), 6),
    (([0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1], 3), 10),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

For every subarray count its zeros; keep the longest with ≤ k zeros.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

The window may contain at most k zeros. Expand r; when zeros exceed k, move l until one zero leaves.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** For every subarray count its zeros; keep the longest with ≤ k zeros.

```python
def longest_ones(nums, k):
    best = 0
    for i in range(len(nums)):
        zeros = 0
        for j in range(i, len(nums)):
            zeros += nums[j] == 0
            if zeros > k:
                break
            best = max(best, j - i + 1)
    return best
```

**Cost:** O(n²) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** The window may contain at most k zeros. Expand r; when zeros exceed k, move l until one zero leaves.

```python
def longest_ones(nums, k):
    l = zeros = best = 0
    for r, x in enumerate(nums):
        zeros += x == 0
        while zeros > k:
            zeros -= nums[l] == 0
            l += 1
        best = max(best, r - l + 1)
    return best
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: For every subarray count its zeros; keep the longest with ≤ k zeros. That costs O(n²) time and O(1) space."

"I can do better: The window may contain at most k zeros. Expand r; when zeros exceed k, move l until one zero leaves. That's O(n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `nums = [1, 0, 1, 1, 0, 0, 1, 1, 0, 1], k = 2`

1. Window has 0 zero(s) ≤ k = 2, length 1
2. Window has 1 zero(s) ≤ k = 2, length 2
3. Window has 1 zero(s) ≤ k = 2, length 3
4. Window has 1 zero(s) ≤ k = 2, length 4
5. Window has 2 zero(s) ≤ k = 2, length 5
6. Window has 2 zero(s) ≤ k = 2, length 4
7. Window has 2 zero(s) ≤ k = 2, length 5
8. Window has 2 zero(s) ≤ k = 2, length 6
9. Window has 2 zero(s) ≤ k = 2, length 4
10. Window has 2 zero(s) ≤ k = 2, length 5

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/sliding-window/max-consecutive-ones-iii)

<a id="q8"></a>
### Q8. Fruit Into Baskets

🟡 Interview standard · *At most 2 distinct*

Walking right, you pick one fruit from every tree but your two baskets each hold one fruit type. Return the most fruit you can collect — the longest subarray with at most 2 distinct values.

**Examples**

- `fruits = [1, 2, 1]` → `3`
- `fruits = [0, 1, 2, 2]` → `3`
- `fruits = [1, 2, 3, 2, 2]` → `4`

In [ ]:
def total_fruit(fruits):
    """
    Walking right, you pick one fruit from every tree but your two baskets each
    hold one fruit type. Return the most fruit you can collect — the longest
    subarray with at most 2 distinct values.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(total_fruit, [
    (([1, 2, 1],), 3),
    (([0, 1, 2, 2],), 3),
    (([1, 2, 3, 2, 2],), 4),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

For every start, extend until a third type appears.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Window with a count map; when it holds 3 types, shrink from the left until a type's count reaches 0 and is removed.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** For every start, extend until a third type appears.

```python
def total_fruit(fruits):
    best = 0
    for i in range(len(fruits)):
        kinds = set()
        for j in range(i, len(fruits)):
            kinds.add(fruits[j])
            if len(kinds) > 2:
                break
            best = max(best, j - i + 1)
    return best
```

**Cost:** O(n²) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Window with a count map; when it holds 3 types, shrink from the left until a type's count reaches 0 and is removed.

```python
def total_fruit(fruits):
    count, l, best = {}, 0, 0
    for r, f in enumerate(fruits):
        count[f] = count.get(f, 0) + 1
        while len(count) > 2:
            count[fruits[l]] -= 1
            if count[fruits[l]] == 0:
                del count[fruits[l]]
            l += 1
        best = max(best, r - l + 1)
    return best
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: For every start, extend until a third type appears. That costs O(n²) time and O(1) space."

"I can do better: Window with a count map; when it holds 3 types, shrink from the left until a type's count reaches 0 and is removed. That's O(n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `fruits = [3, 3, 3, 1, 2, 1, 1, 2, 3, 3, 4]`

1. Types in basket: [3]; window length 1
2. Types in basket: [3]; window length 2
3. Types in basket: [3]; window length 3
4. Types in basket: [3, 1]; window length 4
5. Types in basket: [1, 2]; window length 2
6. Types in basket: [1, 2]; window length 3
7. Types in basket: [1, 2]; window length 4
8. Types in basket: [1, 2]; window length 5
9. Types in basket: [2, 3]; window length 2
10. Types in basket: [2, 3]; window length 3
11. Types in basket: [3, 4]; window length 3

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/sliding-window/fruit-into-baskets)

<a id="q9"></a>
### Q9. Find All Anagrams in a String

🟡 Interview standard · *Fixed window · counts*

Return every start index in `s` where an anagram of `p` begins.

**Examples**

- `s = 'cbaebabacd', p = 'abc'` → `[0, 6]`
- `s = 'abab', p = 'ab'` → `[0, 1, 2]`

In [ ]:
def find_anagrams(s, p):
    """
    Return every start index in s where an anagram of p begins.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(find_anagrams, [
    (('cbaebabacd', 'abc'), [0, 6]),
    (('abab', 'ab'), [0, 1, 2]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Sort each window of length |p| and compare with sorted p.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Slide a window of length |p| with a 26-count array and compare with p's counts at each step.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Sort each window of length |p| and compare with sorted p.

```python
def find_anagrams(s, p):
    m, key = len(p), sorted(p)
    return [i for i in range(len(s) - m + 1) if sorted(s[i:i + m]) == key]
```

**Cost:** O(n · m log m) time, O(m) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Slide a window of length |p| with a 26-count array and compare with p's counts at each step.

```python
def find_anagrams(s, p):
    m, res = len(p), []
    need, win = [0] * 26, [0] * 26
    for ch in p:
        need[ord(ch) - 97] += 1
    for r, ch in enumerate(s):
        win[ord(ch) - 97] += 1
        if r >= m:
            win[ord(s[r - m]) - 97] -= 1
        if r >= m - 1 and win == need:
            res.append(r - m + 1)
    return res
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Sort each window of length |p| and compare with sorted p. That costs O(n · m log m) time and O(m) space."

"I can do better: Slide a window of length |p| with a 26-count array and compare with p's counts at each step. That's O(n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `s = 'cbaebabacd', p = 'abc'`

1. Window 'c'
2. Window 'cb'
3. Window 'cba'
4. Window 'bae'
5. Window 'aeb'
6. Window 'eba'
7. Window 'bab'
8. Window 'aba'
9. Window 'bac'
10. Window 'acd'

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/sliding-window/find-all-anagrams-in-a-string)

<a id="q10"></a>
### Q10. Number of Substrings Containing All Three Characters

🟡 Interview standard · *Variable window · count starts*

`s` contains only a, b and c. Count the substrings that contain at least one of each.

**Examples**

- `s = 'abcabc'` → `10`
- `s = 'aaacb'` → `3`
- `s = 'abc'` → `1`

In [ ]:
def number_of_substrings(s):
    """
    s contains only a, b and c. Count the substrings that contain at least one
    of each.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(number_of_substrings, [
    (('abcabc',), 10),
    (('aaacb',), 3),
    (('abc',), 1),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Check every substring.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

For each r, shrink l while the window still has all three. Every start before l then works with this r, so add l.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Check every substring.

```python
def number_of_substrings(s):
    count = 0
    for i in range(len(s)):
        seen = set()
        for j in range(i, len(s)):
            seen.add(s[j])
            count += len(seen) == 3
    return count
```

**Cost:** O(n²) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** For each r, shrink l while the window still has all three. Every start before l then works with this r, so add l.

```python
def number_of_substrings(s):
    cnt = {"a": 0, "b": 0, "c": 0}
    l = total = 0
    for r, ch in enumerate(s):
        cnt[ch] += 1
        while all(cnt.values()):
            cnt[s[l]] -= 1
            l += 1
        total += l
    return total
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Check every substring. That costs O(n²) time and O(1) space."

"I can do better: For each r, shrink l while the window still has all three. Every start before l then works with this r, so add l. That's O(n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `s = 'abcabc'`

1. Substrings ending at 0 with a, b, c: starts 0..-1 → +0
2. Substrings ending at 1 with a, b, c: starts 0..-1 → +0
3. Substrings ending at 2 with a, b, c: starts 0..0 → +1
4. Substrings ending at 3 with a, b, c: starts 0..1 → +2
5. Substrings ending at 4 with a, b, c: starts 0..2 → +3
6. Substrings ending at 5 with a, b, c: starts 0..3 → +4

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/sliding-window/number-of-substrings-containing-all-three-characters)

<a id="q11"></a>
### Q11. Maximum Points You Can Obtain from Cards

🟡 Interview standard · *Window on the complement*

Take exactly `k` cards, each from the left or right end of the row. Maximise the total points.

**Examples**

- `card_points = [1, 2, 3, 4, 5, 6, 1], k = 3` → `12`
- `card_points = [2, 2, 2], k = 2` → `4`
- `card_points = [9, 7, 7, 9, 7, 7, 9], k = 7` → `55`

In [ ]:
def max_score(card_points, k):
    """
    Take exactly k cards, each from the left or right end of the row. Maximise
    the total points.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(max_score, [
    (([1, 2, 3, 4, 5, 6, 1], 3), 12),
    (([2, 2, 2], 2), 4),
    (([9, 7, 7, 9, 7, 7, 9], 7), 55),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Try taking i from the left and k − i from the right for every i, summing each time.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

The cards you leave are a contiguous window of n − k. Minimise that window's sum with a fixed sliding window; answer = total − min.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Try taking i from the left and k − i from the right for every i, summing each time.

```python
def max_score(card_points, k):
    best = 0
    for i in range(k + 1):
        best = max(best, sum(card_points[:i]) + sum(card_points[len(card_points) - (k - i):]))
    return best
```

**Cost:** O(k²) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** The cards you leave are a contiguous window of n − k. Minimise that window's sum with a fixed sliding window; answer = total − min.

```python
def max_score(card_points, k):
    n = len(card_points)
    w = n - k
    total = sum(card_points)
    cur = low = sum(card_points[:w])
    lo_l = 0
    for r in range(w, n):
        cur += card_points[r] - card_points[r - w]
        if cur < low:
            low, lo_l = cur, r - w + 1
    return total - low
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Try taking i from the left and k − i from the right for every i, summing each time. That costs O(k²) time and O(1) space."

"I can do better: The cards you leave are a contiguous window of n − k. Minimise that window's sum with a fixed sliding window; answer = total − min. That's O(n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `card_points = [1, 2, 3, 4, 5, 6, 1], k = 3`

1. Leave a window of 4 cards; its sum is 10
2. Window 1..4 sums to 14; smallest so far 10
3. Window 2..5 sums to 18; smallest so far 10
4. Window 3..6 sums to 16; smallest so far 10

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/sliding-window/maximum-points-you-can-obtain-from-cards)

<a id="q12"></a>
### Q12. Binary Subarrays With Sum

🟡 Interview standard · *atMost trick on 0/1*

Count subarrays of binary `nums` whose sum equals `goal`.

**Examples**

- `nums = [1, 0, 1, 0, 1], goal = 2` → `4`
- `nums = [0, 0, 0, 0, 0], goal = 0` → `15`

In [ ]:
def num_subarrays_with_sum(nums, goal):
    """
    Count subarrays of binary nums whose sum equals goal.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(num_subarrays_with_sum, [
    (([1, 0, 1, 0, 1], 2), 4),
    (([0, 0, 0, 0, 0], 0), 15),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Sum every subarray.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Count subarrays with sum ≤ goal using a window (r − l + 1 per step), then exactly(goal) = atMost(goal) − atMost(goal − 1).

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Sum every subarray.

```python
def num_subarrays_with_sum(nums, goal):
    count = 0
    for i in range(len(nums)):
        total = 0
        for j in range(i, len(nums)):
            total += nums[j]
            count += total == goal
    return count
```

**Cost:** O(n²) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Count subarrays with sum ≤ goal using a window (r − l + 1 per step), then exactly(goal) = atMost(goal) − atMost(goal − 1).

```python
def num_subarrays_with_sum(nums, goal):
    def at_most(g):
        if g < 0:
            return 0
        l = total = count = 0
        for r, x in enumerate(nums):
            total += x
            while total > g:
                total -= nums[l]
                l += 1
            count += r - l + 1
        return count
    return at_most(goal) - at_most(goal - 1)
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Sum every subarray. That costs O(n²) time and O(1) space."

"I can do better: Count subarrays with sum ≤ goal using a window (r − l + 1 per step), then exactly(goal) = atMost(goal) − atMost(goal − 1). That's O(n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `nums = [1, 0, 1, 0, 1], goal = 2`

1. atMost(2): window sum 1, +1
2. atMost(2): window sum 1, +2
3. atMost(2): window sum 2, +3
4. atMost(2): window sum 2, +4
5. atMost(2): window sum 2, +4
6. atMost(1): window sum 1, +1
7. atMost(1): window sum 1, +2
8. atMost(1): window sum 1, +2
9. atMost(1): window sum 1, +3
10. atMost(1): window sum 1, +2

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/sliding-window/binary-subarrays-with-sum)

<a id="q13"></a>
### Q13. Count Number of Nice Subarrays

🟡 Interview standard · *atMost trick*

A subarray is nice if it contains exactly `k` odd numbers. Count them.

**Examples**

- `nums = [1, 1, 2, 1, 1], k = 3` → `2`
- `nums = [2, 4, 6], k = 1` → `0`
- `nums = [2, 2, 2, 1, 2, 2, 1, 2, 2, 2], k = 2` → `16`

In [ ]:
def number_of_subarrays(nums, k):
    """
    A subarray is nice if it contains exactly k odd numbers. Count them.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(number_of_subarrays, [
    (([1, 1, 2, 1, 1], 3), 2),
    (([2, 4, 6], 1), 0),
    (([2, 2, 2, 1, 2, 2, 1, 2, 2, 2], 2), 16),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Count odds in every subarray.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Same idea as binary subarrays: treat odd as 1, even as 0, and use atMost(k) − atMost(k − 1).

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Count odds in every subarray.

```python
def number_of_subarrays(nums, k):
    count = 0
    for i in range(len(nums)):
        odd = 0
        for j in range(i, len(nums)):
            odd += nums[j] % 2
            count += odd == k
    return count
```

**Cost:** O(n²) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Same idea as binary subarrays: treat odd as 1, even as 0, and use atMost(k) − atMost(k − 1).

```python
def number_of_subarrays(nums, k):
    def at_most(k):
        l = odd = count = 0
        for r, x in enumerate(nums):
            odd += x % 2
            while odd > k:
                odd -= nums[l] % 2
                l += 1
            count += r - l + 1
        return count
    return at_most(k) - at_most(k - 1)
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Count odds in every subarray. That costs O(n²) time and O(1) space."

"I can do better: Same idea as binary subarrays: treat odd as 1, even as 0, and use atMost(k) − atMost(k − 1). That's O(n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `nums = [1, 1, 2, 1, 1], k = 3`

1. atMost(3) odds: window has 1, +1
2. atMost(3) odds: window has 2, +2
3. atMost(3) odds: window has 2, +3
4. atMost(3) odds: window has 3, +4
5. atMost(3) odds: window has 3, +4
6. atMost(2) odds: window has 1, +1
7. atMost(2) odds: window has 2, +2
8. atMost(2) odds: window has 2, +3
9. atMost(2) odds: window has 2, +3
10. atMost(2) odds: window has 2, +3

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/sliding-window/count-number-of-nice-subarrays)

<a id="q14"></a>
### Q14. Maximum Number of Vowels in a Substring of Given Length

🟡 Interview standard · *Fixed window · count*

Return the most vowels in any substring of `s` of length `k`.

**Examples**

- `s = 'abciiidef', k = 3` → `3`
- `s = 'aeiou', k = 2` → `2`
- `s = 'leetcode', k = 3` → `2`

In [ ]:
def max_vowels(s, k):
    """
    Return the most vowels in any substring of s of length k.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(max_vowels, [
    (('abciiidef', 3), 3),
    (('aeiou', 2), 2),
    (('leetcode', 3), 2),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Count vowels in every window from scratch.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Slide a window of length k: +1 if the entering char is a vowel, −1 if the leaving one was.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Count vowels in every window from scratch.

```python
def max_vowels(s, k):
    return max(sum(c in "aeiou" for c in s[i:i + k]) for i in range(len(s) - k + 1))
```

**Cost:** O(n · k) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Slide a window of length k: +1 if the entering char is a vowel, −1 if the leaving one was.

```python
def max_vowels(s, k):
    cur = best = 0
    for r, ch in enumerate(s):
        cur += ch in "aeiou"
        if r >= k:
            cur -= s[r - k] in "aeiou"
        best = max(best, cur)
    return best
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Count vowels in every window from scratch. That costs O(n · k) time and O(1) space."

"I can do better: Slide a window of length k: +1 if the entering char is a vowel, −1 if the leaving one was. That's O(n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `s = 'abciiidef', k = 3`

1. Window 'a' has 1 vowel(s)
2. Window 'ab' has 1 vowel(s)
3. Window 'abc' has 1 vowel(s)
4. Window 'bci' has 1 vowel(s)
5. Window 'cii' has 2 vowel(s)
6. Window 'iii' has 3 vowel(s)
7. Window 'iid' has 2 vowel(s)
8. Window 'ide' has 2 vowel(s)
9. Window 'def' has 1 vowel(s)

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/sliding-window/maximum-number-of-vowels-in-a-substring-of-given-length)

<a id="q15"></a>
### Q15. Grumpy Bookstore Owner

🟡 Interview standard · *Fixed window · extra gain*

In minute i, `customers[i]` arrive; they're unhappy if `grumpy[i] = 1`. The owner can stay calm for one stretch of `minutes`. Maximise satisfied customers.

**Examples**

- `customers = [1, 0, 1, 2, 1, 1, 7, 5], grumpy = [0, 1, 0, 1, 0, 1, 0, 1], minutes = 3` → `16`
- `customers = [1], grumpy = [0], minutes = 1` → `1`

In [ ]:
def max_satisfied(customers, grumpy, minutes):
    """
    In minute i, customers[i] arrive; they're unhappy if grumpy[i] = 1. The
    owner can stay calm for one stretch of minutes. Maximise satisfied
    customers.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(max_satisfied, [
    (([1, 0, 1, 2, 1, 1, 7, 5], [0, 1, 0, 1, 0, 1, 0, 1], 3), 16),
    (([1], [0], 1), 1),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Try each calm window and recount all satisfied customers.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Customers in non-grumpy minutes are always happy. The calm window only adds customers in grumpy minutes, so find the window of length minutes with the largest such sum.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Try each calm window and recount all satisfied customers.

```python
def max_satisfied(customers, grumpy, minutes):
    best, n = 0, len(customers)
    for s in range(n - minutes + 1):
        total = sum(customers[i] for i in range(n) if not grumpy[i] or s <= i < s + minutes)
        best = max(best, total)
    return best
```

**Cost:** O(n · m) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Customers in non-grumpy minutes are always happy. The calm window only adds customers in grumpy minutes, so find the window of length minutes with the largest such sum.

```python
def max_satisfied(customers, grumpy, minutes):
    base = sum(c for c, g in zip(customers, grumpy) if not g)
    extra = best = 0
    for r in range(len(customers)):
        extra += customers[r] * grumpy[r]
        if r >= minutes:
            extra -= customers[r - minutes] * grumpy[r - minutes]
        best = max(best, extra)
    return base + best
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Try each calm window and recount all satisfied customers. That costs O(n · m) time and O(1) space."

"I can do better: Customers in non-grumpy minutes are always happy. The calm window only adds customers in grumpy minutes, so find the window of length minutes with the largest such sum. That's O(n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `customers = [1, 0, 1, 2, 1, 1, 7, 5], grumpy = [0, 1, 0, 1, 0, 1, 0, 1], minutes = 3`

1. Calm window ending at 0 rescues 0 customers
2. Calm window ending at 1 rescues 0 customers
3. Calm window ending at 2 rescues 0 customers
4. Calm window ending at 3 rescues 2 customers
5. Calm window ending at 4 rescues 2 customers
6. Calm window ending at 5 rescues 3 customers
7. Calm window ending at 6 rescues 1 customers
8. Calm window ending at 7 rescues 6 customers

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/sliding-window/grumpy-bookstore-owner)

<a id="q16"></a>
### Q16. Longest Subarray of 1's After Deleting One Element

🟡 Interview standard · *At most one zero*

Delete exactly one element from binary `nums`. Return the longest run of 1s left.

**Examples**

- `nums = [1, 1, 0, 1]` → `3`
- `nums = [0, 1, 1, 1, 0, 1, 1, 0, 1]` → `5`
- `nums = [1, 1, 1]` → `2`

In [ ]:
def longest_subarray(nums):
    """
    Delete exactly one element from binary nums. Return the longest run of 1s
    left.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(longest_subarray, [
    (([1, 1, 0, 1],), 3),
    (([0, 1, 1, 1, 0, 1, 1, 0, 1],), 5),
    (([1, 1, 1],), 2),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Delete each index in turn and measure the longest run.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Longest window with at most one zero, minus one (the deleted element).

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Delete each index in turn and measure the longest run.

```python
def longest_subarray(nums):
    best = 0
    for d in range(len(nums)):
        rest, run = nums[:d] + nums[d + 1:], 0
        for x in rest:
            run = run + 1 if x else 0
            best = max(best, run)
    return best
```

**Cost:** O(n²) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Longest window with at most one zero, minus one (the deleted element).

```python
def longest_subarray(nums):
    l = zeros = best = 0
    for r, x in enumerate(nums):
        zeros += x == 0
        while zeros > 1:
            zeros -= nums[l] == 0
            l += 1
        best = max(best, r - l)
    return best
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Delete each index in turn and measure the longest run. That costs O(n²) time and O(n) space."

"I can do better: Longest window with at most one zero, minus one (the deleted element). That's O(n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `nums = [1, 1, 0, 1]`

1. Window 0..0 has 0 zero(s); after deleting one element: 0
2. Window 0..1 has 0 zero(s); after deleting one element: 1
3. Window 0..2 has 1 zero(s); after deleting one element: 2
4. Window 0..3 has 1 zero(s); after deleting one element: 3

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/sliding-window/longest-subarray-of-1s-after-deleting-one-element)

<a id="q17"></a>
### Q17. Frequency of the Most Frequent Element

🟡 Interview standard · *Sort + window cost*

In one operation you add 1 to any element. With at most `k` operations, return the highest possible frequency of one value.

**Examples**

- `nums = [1, 2, 4], k = 5` → `3`
- `nums = [1, 4, 8, 13], k = 5` → `2`
- `nums = [3, 9, 6], k = 2` → `1`

In [ ]:
def max_frequency(nums, k):
    """
    In one operation you add 1 to any element. With at most k operations, return
    the highest possible frequency of one value.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(max_frequency, [
    (([1, 2, 4], 5), 3),
    (([1, 4, 8, 13], 5), 2),
    (([3, 9, 6], 2), 1),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

For each target value, greedily raise the closest smaller values.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Sort. In window [l, r] raise everything to nums[r]; cost = nums[r] · len − windowSum. Shrink while cost &gt; k.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** For each target value, greedily raise the closest smaller values.

```python
def max_frequency(nums, k):
    nums = sorted(nums)
    best = 1
    for r in range(len(nums)):
        budget, l = k, r
        while l > 0 and budget >= nums[r] - nums[l - 1]:
            budget -= nums[r] - nums[l - 1]
            l -= 1
        best = max(best, r - l + 1)
    return best
```

**Cost:** O(n²) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Sort. In window [l, r] raise everything to nums[r]; cost = nums[r] · len − windowSum. Shrink while cost &gt; k.

```python
def max_frequency(nums, k):
    nums.sort()
    l = total = best = 0
    for r, x in enumerate(nums):
        total += x
        while x * (r - l + 1) - total > k:
            total -= nums[l]
            l += 1
        best = max(best, r - l + 1)
    return best
```

**Cost:** O(n log n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: For each target value, greedily raise the closest smaller values. That costs O(n²) time and O(1) space."

"I can do better: Sort. In window [l, r] raise everything to nums[r]; cost = nums[r] · len − windowSum. Shrink while cost &gt; k. That's O(n log n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `nums = [1, 2, 4], k = 5`

1. Raise window to 1: cost 1 × 1 − 1 = 0 ≤ 5
2. Raise window to 2: cost 2 × 2 − 3 = 1 ≤ 5
3. Raise window to 4: cost 4 × 3 − 7 = 5 ≤ 5

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/sliding-window/frequency-of-the-most-frequent-element)

<a id="q18"></a>
### Q18. Minimum Window Substring

🔴 Stretch · *Variable window · need / have*

Return the shortest substring of `s` containing every character of `t` (with multiplicity), or `""`.

**Examples**

- `s = 'ADOBECODEBANC', t = 'ABC'` → `'BANC'`
- `s = 'a', t = 'a'` → `'a'`
- `s = 'a', t = 'aa'` → `''`

In [ ]:
def min_window(s, t):
    """
    Return the shortest substring of s containing every character of t (with
    multiplicity), or "".
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(min_window, [
    (('ADOBECODEBANC', 'ABC'), 'BANC'),
    (('a', 'a'), 'a'),
    (('a', 'aa'), ''),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Check every substring of s for containing t's counts.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Expand r until the window covers t (have == need). Then shrink l as long as it still covers, recording the smallest. Track how many distinct letters are satisfied so the check is O(1).

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Check every substring of s for containing t's counts.

```python
def min_window(s, t):
    from collections import Counter
    need, best = Counter(t), ""
    for i in range(len(s)):
        for j in range(i, len(s)):
            if not need - Counter(s[i:j + 1]):
                if not best or j - i + 1 < len(best):
                    best = s[i:j + 1]
                break
    return best
```

**Cost:** O(n² · σ) time, O(σ) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Expand r until the window covers t (have == need). Then shrink l as long as it still covers, recording the smallest. Track how many distinct letters are satisfied so the check is O(1).

```python
def min_window(s, t):
    need = {}
    for ch in t:
        need[ch] = need.get(ch, 0) + 1
    have, required = 0, len(need)
    win, l, best = {}, 0, (float("inf"), 0, 0)
    for r, ch in enumerate(s):
        win[ch] = win.get(ch, 0) + 1
        if ch in need and win[ch] == need[ch]:
            have += 1
        while have == required:
            if r - l + 1 < best[0]:
                best = (r - l + 1, l, r)
            win[s[l]] -= 1
            if s[l] in need and win[s[l]] < need[s[l]]:
                have -= 1
            l += 1
    return "" if best[0] == float("inf") else s[best[1]:best[2] + 1]
```

**Cost:** O(n + m) time, O(σ) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Check every substring of s for containing t's counts. That costs O(n² · σ) time and O(σ) space."

"I can do better: Expand r until the window covers t (have == need). Then shrink l as long as it still covers, recording the smallest. Track how many distinct letters are satisfied so the check is O(1). That's O(n + m) time and O(σ) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `s = 'ADOBECODEBANC', t = 'ABC'`

1. Window 'ADOBEC' covers 'ABC' → try shrinking from the left
2. Window 'DOBECODEBA' covers 'ABC' → try shrinking from the left
3. Window 'OBECODEBA' covers 'ABC' → try shrinking from the left
4. Window 'BECODEBA' covers 'ABC' → try shrinking from the left
5. Window 'ECODEBA' covers 'ABC' → try shrinking from the left
6. Window 'CODEBA' covers 'ABC' → try shrinking from the left
7. Window 'ODEBANC' covers 'ABC' → try shrinking from the left
8. Window 'DEBANC' covers 'ABC' → try shrinking from the left
9. Window 'EBANC' covers 'ABC' → try shrinking from the left
10. Window 'BANC' covers 'ABC' → try shrinking from the left

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/sliding-window/minimum-window-substring)

<a id="q19"></a>
### Q19. Sliding Window Maximum

🔴 Stretch · *Monotonic deque*

Return the maximum of every window of size `k` as it slides across `nums`.

**Examples**

- `nums = [1, 3, -1, -3, 5, 3, 6, 7], k = 3` → `[3, 3, 5, 5, 6, 7]`
- `nums = [1], k = 1` → `[1]`

In [ ]:
def max_sliding_window(nums, k):
    """
    Return the maximum of every window of size k as it slides across nums.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(max_sliding_window, [
    (([1, 3, -1, -3, 5, 3, 6, 7], 3), [3, 3, 5, 5, 6, 7]),
    (([1], 1), [1]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Compute max() of every window.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Keep a deque of indices whose values are decreasing. Pop smaller values from the back before pushing (they can never be a max again); pop the front when it leaves the window. The front is always the max.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Compute max() of every window.

```python
def max_sliding_window(nums, k):
    return [max(nums[i:i + k]) for i in range(len(nums) - k + 1)]
```

**Cost:** O(n · k) time, O(1) extra space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Keep a deque of indices whose values are decreasing. Pop smaller values from the back before pushing (they can never be a max again); pop the front when it leaves the window. The front is always the max.

```python
def max_sliding_window(nums, k):
    from collections import deque
    dq, res = deque(), []
    for r, x in enumerate(nums):
        while dq and nums[dq[-1]] <= x:
            dq.pop()
        dq.append(r)
        if dq[0] <= r - k:
            dq.popleft()
        if r >= k - 1:
            res.append(nums[dq[0]])
    return res
```

**Cost:** O(n) time, O(k) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Compute max() of every window. That costs O(n · k) time and O(1) extra space."

"I can do better: Keep a deque of indices whose values are decreasing. Pop smaller values from the back before pushing (they can never be a max again); pop the front when it leaves the window. The front is always the max. That's O(n) time and O(k) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `nums = [1, 3, -1, -3, 5, 3, 6, 7], k = 3`

1. Deque (indices, values decreasing): front 0 → max 1
2. Deque (indices, values decreasing): front 1 → max 3
3. Deque (indices, values decreasing): front 1 → max 3
4. Deque (indices, values decreasing): front 1 → max 3
5. Deque (indices, values decreasing): front 4 → max 5
6. Deque (indices, values decreasing): front 4 → max 5
7. Deque (indices, values decreasing): front 6 → max 6
8. Deque (indices, values decreasing): front 7 → max 7

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/sliding-window/sliding-window-maximum)

<a id="q20"></a>
### Q20. Subarrays with K Different Integers

🔴 Stretch · *exactly = atMost(k) − atMost(k−1)*

Return the number of subarrays with exactly `k` distinct integers.

**Examples**

- `nums = [1, 2, 1, 2, 3], k = 2` → `7`
- `nums = [1, 2, 1, 3, 4], k = 3` → `3`

In [ ]:
def subarrays_with_k_distinct(nums, k):
    """
    Return the number of subarrays with exactly k distinct integers.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(subarrays_with_k_distinct, [
    (([1, 2, 1, 2, 3], 2), 7),
    (([1, 2, 1, 3, 4], 3), 3),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Enumerate every subarray and count its distinct values.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Counting 'at most k distinct' is easy with a window: each r adds r − l + 1 subarrays. Exactly k = atMost(k) − atMost(k − 1).

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Enumerate every subarray and count its distinct values.

```python
def subarrays_with_k_distinct(nums, k):
    count = 0
    for i in range(len(nums)):
        seen = set()
        for j in range(i, len(nums)):
            seen.add(nums[j])
            count += len(seen) == k
    return count
```

**Cost:** O(n²) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Counting 'at most k distinct' is easy with a window: each r adds r − l + 1 subarrays. Exactly k = atMost(k) − atMost(k − 1).

```python
def subarrays_with_k_distinct(nums, k):
    def at_most(k):
        count, l, total = {}, 0, 0
        for r, x in enumerate(nums):
            count[x] = count.get(x, 0) + 1
            while len(count) > k:
                count[nums[l]] -= 1
                if count[nums[l]] == 0:
                    del count[nums[l]]
                l += 1
            total += r - l + 1
        return total
    return at_most(k) - at_most(k - 1)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Enumerate every subarray and count its distinct values. That costs O(n²) time and O(n) space."

"I can do better: Counting 'at most k distinct' is easy with a window: each r adds r − l + 1 subarrays. Exactly k = atMost(k) − atMost(k − 1). That's O(n) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `nums = [1, 2, 1, 2, 3], k = 2`

1. atMost(2): 1 subarrays end at 0
2. atMost(2): 2 subarrays end at 1
3. atMost(2): 3 subarrays end at 2
4. atMost(2): 4 subarrays end at 3
5. atMost(2): 2 subarrays end at 4
6. atMost(1): 1 subarrays end at 0
7. atMost(1): 1 subarrays end at 1
8. atMost(1): 1 subarrays end at 2
9. atMost(1): 1 subarrays end at 3
10. atMost(1): 1 subarrays end at 4

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/sliding-window/subarrays-with-k-different-integers)

---
# Part 4: Interview questions, out loud

Cover the answer, **say yours out loud**, then open it and compare. Aim for two or three clear sentences, then an example.

<details>
<summary><b>1. How is a sliding window O(n) when it has a nested loop?</b></summary>

Each pointer moves only forward and at most n steps. The inner `while` loop's total iterations across the entire run are bounded by n, so the total work is O(2n) = O(n).

</details>

<details>
<summary><b>2. Why doesn't sliding window work with negative numbers for 'sum = k'?</b></summary>

With negatives, shrinking the window doesn't reliably decrease the sum and expanding doesn't reliably increase it, so there's no correct rule for when to move l. Prefix sums with a hash map of counts solve it in O(n).

</details>

<details>
<summary><b>3. When do you update the answer — before or after shrinking?</b></summary>

For the **longest** valid window, shrink until valid, then update. For the **shortest** valid window, update while it is still valid, then shrink.

</details>

<details>
<summary><b>4. How do you count subarrays with exactly K distinct elements?</b></summary>

Counting with 'at most K' is easy: after shrinking, every start between l and r is valid, adding r − l + 1. Then exactly(K) = atMost(K) − atMost(K − 1).

</details>

<details>
<summary><b>5. How do you keep the maximum of a sliding window in O(1)?</b></summary>

A monotonic deque of indices with decreasing values: pop smaller values from the back before pushing, pop the front when it leaves the window. The front is the max. Each index is pushed and popped once.

</details>


---
# Check yourself before moving on

You're ready for the next notebook when you can tick these **without looking**:

- [ ] I can explain: Subarrays, and why brute force wastes work
- [ ] I can explain: Variable-size window: expand, then shrink
- [ ] I can explain: Windows that count characters
- [ ] I can explain: Counting subarrays: the atMost trick
- [ ] I can explain: Window maximum with a monotonic deque

**Revision drill.** Come back in a few days, set a 25-minute timer per question, and solve these from a blank cell:

- [ ] [Q16](#q16) Longest Subarray of 1's After Deleting One Element
- [ ] [Q17](#q17) Frequency of the Most Frequent Element
- [ ] [Q18](#q18) Minimum Window Substring
- [ ] [Q19](#q19) Sliding Window Maximum
- [ ] [Q20](#q20) Subarrays with K Different Integers

Spaced repetition beats re-reading: redoing a question after you've partly forgotten it is what makes it stick.